In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

train_images, test_images = train_images / 255.0, test_images / 255.0

callbacks_list = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3)
]

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 12s 0us/step


In [2]:
model_B_scratch = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10)
])

model_B_scratch.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

history_scratch = model_B_scratch.fit(train_images, train_labels, epochs=20,
                    validation_data=(test_images, test_labels),
                    callbacks=callbacks_list)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 303s 192ms/step - accuracy: 0.3884 - loss: 1.6324 - val_accuracy: 0.5728 - val_loss: 1.1974 - learning_rate: 0.0010
Epoch 2/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 300s 192ms/step - accuracy: 0.5255 - loss: 1.2706 - val_accuracy: 0.5596 - val_loss: 1.2202 - learning_rate: 0.0010
Epoch 3/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 320s 191ms/step - accuracy: 0.6003 - loss: 1.0930 - val_accuracy: 0.6847 - val_loss: 0.9075 - learning_rate: 0.0010
Epoch 4/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 330s 196ms/step - accuracy: 0.6576 - loss: 0.9514 - val_accuracy: 0.7028 - val_loss: 0.8717 - learning_rate: 0.0010
Epoch 5/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 323s 197ms/step - accuracy: 0.6991 - loss: 0.8420 - val_accuracy: 0.7096 - val_loss: 0.8571 - learning_rate: 0.0010
Epoch 6/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 306s 196ms/step - accuracy: 0.7306 - loss: 0.7549 - val_accuracy: 0.7064 - val_loss: 0.8607 - learning_rate: 0.0010
Epoch 7/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 301s 193ms

In [3]:
base_model = tf.keras.applications.MobileNetV2(input_shape=(96, 96, 3),
                                               include_top=False,
                                               weights='imagenet')
base_model.trainable = False

model_B_transfer = models.Sequential([
    layers.UpSampling2D((3,3), input_shape=(32, 32, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10)
])

model_B_transfer.compile(optimizer='adam',
                loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=['accuracy'])

model_B_transfer.fit(train_images, train_labels, epochs=5,
                       validation_data=(test_images, test_labels),
                       callbacks=callbacks_list)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/up_sampling2d.py:72: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 408s 257ms/step - accuracy: 0.6973 - loss: 0.8831 - val_accuracy: 0.7617 - val_loss: 0.6810 - learning_rate: 0.0010
Epoch 2/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 385s 247ms/step - accuracy: 0.7556 - loss: 0.7049 - val_accuracy: 0.7831 - val_loss: 0.6308 - learning_rate: 0.0010
Epoch 3/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 442s 246ms/step - accuracy: 0.7753 - loss: 0.6438 - val_accuracy: 0.7861 - val_loss: 0.6144 - learning_rate: 0.0010
Epoch 4/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 455s 255ms/step - accuracy: 0.7871 - loss: 0.6102 - val_accuracy: 0.7945 - val_loss: 0.6023 - learning_rate: 0.0010
Epoch 5/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 367s 235ms/step - accuracy: 0.7976 - loss: 0.5752 - val_accuracy: 0.7916 - val_loss: 0.6101 - learning_rate: 0.0010


In [4]:
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model_B_transfer.compile(optimizer=tf.keras.optimizers.Adam(0.00001),
                loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=['accuracy'])

history_transfer = model_B_transfer.fit(train_images, train_labels, epochs=15,
                       validation_data=(test_images, test_labels),
                       callbacks=callbacks_list)

Epoch 1/15
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 708s 444ms/step - accuracy: 0.6732 - loss: 1.1773 - val_accuracy: 0.7877 - val_loss: 0.6945 - learning_rate: 1.0000e-05
Epoch 2/15
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 712s 456ms/step - accuracy: 0.7750 - loss: 0.7002 - val_accuracy: 0.8204 - val_loss: 0.5551 - learning_rate: 1.0000e-05
Epoch 3/15
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 689s 441ms/step - accuracy: 0.8119 - loss: 0.5691 - val_accuracy: 0.8319 - val_loss: 0.5027 - learning_rate: 1.0000e-05
Epoch 4/15
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 687s 440ms/step - accuracy: 0.8343 - loss: 0.4844 - val_accuracy: 0.8450 - val_loss: 0.4616 - learning_rate: 1.0000e-05
Epoch 5/15
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 763s 454ms/step - accuracy: 0.8550 - loss: 0.4213 - val_accuracy: 0.8540 - val_loss: 0.4439 - learning_rate: 1.0000e-05
Epoch 6/15
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 711s 455ms/step - accuracy: 0.8691 - loss: 0.3774 - val_accuracy: 0.8607 - val_loss: 0.4269 - learning_rate: 1.0000e-05
Epoch 7/15
1563/1563 ━━━━━━━

In [8]:
loss_s, acc_s = model_B_scratch.evaluate(test_images, test_labels, verbose=0)
loss_t, acc_t = model_B_transfer.evaluate(test_images, test_labels, verbose=0)

print(f"Model B Scratch Accuracy: {acc_s:.4f}")
print(f"Model B Transfer Accuracy: {acc_t:.4f}")

Model B Scratch Accuracy: 0.7904
Model B Transfer Accuracy: 0.8727
